In [ ]:
import numpy as np
import time
import numba
import math
from numpy.random import Generator, PCG64DXSM, SeedSequence
import concurrent.futures
from structure_ORIGINAL_FNs_singlesideBASEreinLEARNING2_2026_updated import play_sequence

np.set_printoptions(suppress=True)

In [ ]:
# Parameters
sides = 1
pairs = np.asarray([[0, 1], [1, 0], [1, 2], [2, 1], [2, 3], [3, 2], [3, 4], [4, 3]], dtype=np.int64)
testpairs = np.asarray([[1, 3], [3, 1]], dtype=np.int64)
nonadjpairs = np.asarray([[0, 2], [2, 0], [0, 3], [3, 0], [0, 4], [4, 0], [1, 4], [4, 1], [2, 4], [4, 2]], dtype=np.int64)
allpairs = np.concatenate((pairs, nonadjpairs, testpairs))

def make_linear_rewards(p):
    return np.asarray([[1, 0] if a > b else [0, 1] for a, b in p], dtype=np.int64)

pairs_reward = make_linear_rewards(pairs)
testpairs_reward = make_linear_rewards(testpairs)
nonadjpairs_reward = make_linear_rewards(nonadjpairs)
allpairs_reward = np.concatenate((pairs_reward, nonadjpairs_reward, testpairs_reward))

plen = len(pairs)
alen = len(allpairs)

terms = 5
maxvalue = 10
startstop = 2
noise = 0.
annealing = 0.
timesteps = 10**8
runs = 1000
rein1, rein2, punish1, punish2 = 4, 4, -11, -11
inertia = 1
nsteps = 100
blocklength = nsteps * 1
thrds = 3

In [ ]:
start = time.perf_counter()

sq1 = SeedSequence()
randomentropy = sq1.entropy
sg = SeedSequence(randomentropy)
rgs = [Generator(PCG64DXSM(s)) for s in sg.spawn(runs)]

final_sigweights = np.zeros((runs, sides, 2, terms, maxvalue))
final_cumsuc = np.zeros(runs)
final_cumsucadd = np.zeros(runs)
final_testcumsucadd = np.zeros(runs)
final_recweights = np.zeros((runs, (maxvalue+1)**2, startstop))
final_pair_stats = np.zeros((runs, len(testpairs), 2), dtype=np.int64)

with concurrent.futures.ProcessPoolExecutor(max_workers=thrds) as executor:
    future_to_run = {executor.submit(play_sequence, r, rgs[r], rein1, punish1, rein2, punish2, timesteps, nsteps, sides, pairs, testpairs, nonadjpairs, allpairs, pairs_reward, testpairs_reward, nonadjpairs_reward, allpairs_reward, plen, alen, terms, maxvalue, startstop, noise, annealing, inertia, blocklength): r for r in range(runs)}
    for future in concurrent.futures.as_completed(future_to_run):
        run_id = future_to_run[future]
        try:
            rid, sigw, cumsuc, cumsucadd, testcumsuc, recw, pstats = future.result()
            final_sigweights[rid] = sigw
            final_cumsuc[rid] = cumsuc
            final_cumsucadd[rid] = cumsucadd
            final_testcumsucadd[rid] = testcumsuc
            final_recweights[rid] = recw
            final_pair_stats[rid] = pstats
        except Exception as exc:
            print(f'generated an exception for run {run_id}: {exc}')
        if run_id % 10 == 0:
            print(f'finished run #{run_id}')

print(f"average cumsuc = {np.average(final_cumsuc)/runs}")
print(f"average final cumsucadd = {np.average(final_cumsucadd)/nsteps}")
print(f"average test cumsum = {np.average(final_testcumsucadd)/nsteps}")

finish = time.perf_counter()
print(f'Finished in {round(finish-start,0)/60} minutes')

In [ ]:
np.save('structure_noiseAnn_dsktp_1s_BASElearning-MV10_updated_sigweights', final_sigweights)
np.save('structure_noiseAnn_dsktp_1s_BASElearning-MV10_updated_recweights', final_recweights)
np.save('structure_noiseAnn_dsktp_1s_BASElearning-MV10_updated_testcumsucadd', final_testcumsucadd)
np.save('structure_noiseAnn_dsktp_1s_BASElearning-MV10_updated_pair_stats', final_pair_stats)

In [ ]:
cutoff = 90
final_test_count = np.sum(final_testcumsucadd > cutoff)
print(final_test_count)

In [ ]:
@numba.njit
def duplicates(dbins):
    dup = 0
    for iii in range(len(dbins)):
        for jjj in range(len(dbins)):
            if iii != jjj:
                if (dbins[iii][0] == dbins[jjj][0]) & (dbins[iii][1] == dbins[jjj][1]):
                    dup = 1
    return dup

@numba.njit
def runs_dup_bins(runs, fsigweights, t):
    dup_count = 0
    for i0 in range(runs):
        sw = fsigweights[i0].copy()
        swl = sw.argmax(axis=3)[0, 0]
        swr = sw.argmax(axis=3)[0, 1]
        bns = np.zeros(((2*(t-1)), 2), dtype = np.int64)
        for idx000 in range(t-1):
            bns[idx000][0] = swl[idx000] 
            bns[idx000][1] = swr[idx000+1]
        for idx001 in range(t-1):
            bns[t-1+idx001][0] = swl[idx001+1] 
            bns[t-1+idx001][1] = swr[idx001]
        dup_count += duplicates(bns)
    return dup_count

In [ ]:
print(runs_dup_bins(runs, final_sigweights, terms))